# Clinical Strand — Data Exploration

**MultimodalAI‧26 · Clinical Demo**

---

Profile the patient dataset. By the end, you should understand:
- Who is in this dataset and which subgroups are at highest risk
- Why clinical notes are Missing Not At Random (MNAR)
- Why blood loss missingness is also MNAR
- Which subgroups are most at risk of being under-served by the models

**Sections**
1. Setup & Load Data
2. Patient Demographics & Severity
3. Clinical Notes — MNAR Analysis
4. Blood Loss — MNAR Analysis
5. Vital Signs Overview
6. Data Profile Verdict

---
## Section 1 — Setup & Load Data

The dataset contains **500 ICU patients, one row per patient** (tabular.csv).

Primary outcome: `major_complication_30d` (1 = major complication within 30 days, ~25%).

Key modalities:
- **Demographics**: age, sex, surgery type, admission urgency
- **Severity**: SOFA score, ASA class
- **Vital signs aggregates**: heart rate, respiratory rate, SpO₂, blood pressure, temperature
- **Laboratory values**: lactate, creatinine, WBC, bilirubin
- **Clinical notes**: NLP-derived risk score (MNAR — missing for ~31% of patients)
- **Blood loss**: intra-operative blood loss (MNAR — missing for ~34% of patients)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / 'data').exists() and (candidate / 'models').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not find clinical_strand root. Run from notebooks/ or clinical_strand/.')

RAW_DATA = ROOT / 'data' / 'raw' / 'tabular.csv'
if not RAW_DATA.exists():
    raise FileNotFoundError(f'{RAW_DATA} not found. Data files are pre-included in the repository.')
print(f'Loading from: {RAW_DATA}')

In [ ]:
df = pd.read_csv(RAW_DATA)
print(f'Shape: {df.shape}')
print(f'\nKey statistics:')
print(f'  Complication rate (30d) : {df["major_complication_30d"].mean():.1%}')
print(f'  Deterioration rate (24h): {df["deteriorated_24h"].mean():.1%}')
print(f'  Patients with notes     : {df["has_notes"].mean():.1%}')
print(f'  Blood loss MNAR         : {df["blood_loss_missing"].mean():.1%}')
print(f'  Age mean / std          : {df["age"].mean():.1f} / {df["age"].std():.1f}')
print(f'  SOFA mean / std         : {df["sofa_score"].mean():.1f} / {df["sofa_score"].std():.1f}')
print(f'\nSurgery type breakdown:')
print(df['surgery_type'].value_counts().to_string())

**Expected output — Section 1:**

```
Loading from: .../data/raw/tabular.csv
Shape: (500, 37)

Key statistics:
  Complication rate (30d) : 25.4%
  Deterioration rate (24h): 40.8%
  Patients with notes     : 68.8%
  Blood loss MNAR         : 33.8%
  Age mean / std          : 62.7 / 13.9
  SOFA mean / std         : 6.8 / 3.3

Surgery type breakdown:
abdominal      155
orthopaedic    140
cardiac        112
vascular        93
```

> Four surgery types. Cardiac is the smallest group by count but has the highest complication rate — relevant for subgroup equity analysis.

---
## Section 2 — Patient Demographics & Severity

Three splits matter for the deployment verdict:
- **Age band**: elderly patients (75+) show blunted physiological responses — heart rate
  may not rise even under haemodynamic stress (beta-blockade, age-related attenuation).
- **Surgery type**: cardiac and vascular carry the highest complication rates. Vascular
  patients receive post-operative beta-blockade which masks vital sign changes.
- **SOFA score**: the primary clinical severity marker. Model failures at extreme SOFA
  scores are a governance risk.

In [ ]:
df['age_band'] = pd.cut(
    df['age'],
    bins=[0, 64, 74, 200],
    labels=['under 65', '65–74', '75+']
)

print('Complication rate by age band:')
print(
    df.groupby('age_band')['major_complication_30d']
    .agg(['mean', 'count']).rename(columns={'mean': 'rate'})
    .round(3).to_string()
)
print()
print('Complication rate by surgery type:')
print(
    df.groupby('surgery_type')['major_complication_30d']
    .agg(['mean', 'count']).rename(columns={'mean': 'rate'})
    .sort_values('rate', ascending=False).round(3).to_string()
)

**Expected output — Section 2 (tables):**

```
Complication rate by age band:
           rate  count
under 65  0.209    273
65-74     0.300    130
75+       0.320     97

Complication rate by surgery type:
               rate  count
cardiac       0.616    112   ← highest risk
vascular      0.258     93
abdominal     0.161    155
orthopaedic   0.064    140   ← lowest risk
```

The bar chart shows the age distribution (red dashed line at 75), complication rate by surgery type, and SOFA score histograms split by outcome. Expect the complication group to shift rightward on the SOFA axis.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['age'].hist(ax=axes[0], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(75, color='red', linestyle='--', linewidth=1.5, label='75 yrs')
axes[0].set_title('Age distribution'); axes[0].legend(fontsize=8)

comp_by_surg = df.groupby('surgery_type')['major_complication_30d'].mean().sort_values(ascending=False)
comp_by_surg.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Complication rate\nby surgery type')
axes[1].set_ylabel('Rate'); axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))

for val, colour, label in [(0, 'steelblue', 'No complication'), (1, 'tomato', 'Complication')]:
    df[df['major_complication_30d'] == val]['sofa_score'].hist(
        ax=axes[2], bins=15, alpha=0.6, color=colour, edgecolor='white', label=label
    )
axes[2].set_title('SOFA score by outcome'); axes[2].legend(fontsize=8)

plt.suptitle('Patient demographics and severity overview', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 3 — Clinical Notes: MNAR Analysis

Clinical notes (`note_risk_score`) are missing for ~**31% of patients**.
This missingness is **not random** — it is Missing Not At Random (MNAR).

Note absence is driven by workload and case complexity:
- Cardiac surgery: highest missing rate (~58%)
- Vascular: second highest (~42%)
- Patients without notes have **higher risk** — the sickest are least documented

**Model C** uses note text as a primary modality. When notes are absent, the text branch
receives a zero vector, causing systematic underestimation of risk for note-absent patients —
concentrated in complex cardiac/vascular cases.

In [ ]:
miss_by_surg = df.groupby('surgery_type')['note_risk_score'].apply(
    lambda x: x.isnull().mean()).rename('missing_rate')
comp_by_notes = df.groupby('has_notes')['major_complication_30d'].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

miss_by_surg.sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Note missingness by surgery type')
axes[0].set_ylabel('Missing rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].tick_params(axis='x', rotation=20)

comp_by_notes.index = ['No notes', 'Has notes']
comp_by_notes.plot(kind='bar', ax=axes[1], color=['tomato', 'seagreen'], edgecolor='white')
axes[1].set_title('Complication rate\nvs notes availability')
axes[1].set_ylabel('Complication rate')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Note missingness is MNAR — absence is NOT a signal of lower risk', fontsize=11)
plt.tight_layout(); plt.show()

print('Note missingness by surgery type:')
for surg, rate in miss_by_surg.sort_values(ascending=False).items():
    print(f'  {surg:14s}: {rate:.1%} missing')
print()
print(f'Complication WITHOUT notes: {comp_by_notes["No notes"]:.1%}')
print(f'Complication WITH notes:    {comp_by_notes["Has notes"]:.1%}')

**Expected output — Section 3:**

```
Note missingness by surgery type:
  cardiac     : 58.0% missing   ← by far the worst
  vascular    : 41.9% missing
  abdominal   : 20.6% missing
  orthopaedic : 14.3% missing

Complication WITHOUT notes: 38.5%
Complication WITH notes:    19.5%
```

The chart on the left shows note missingness by surgery type (bars). The chart on the right shows complication rates for note-absent vs note-present patients — note-absent patients have **double the complication rate**. This confirms that note absence is a proxy for severity, not documentation quality.

---
## Section 4 — Blood Loss: MNAR Analysis

`blood_loss_ml` is missing for ~**34% of patients** — also MNAR.
Missingness is driven by documentation burden during complex cases:
- Cardiac surgery: highest missing rate (~51%)
- Vascular: second highest (~45%)

**Key property:** patients with missing blood loss have **higher** true blood loss on average.
The dataset includes `blood_loss_imputed` (mean-filled), which systematically underestimates
risk for cardiac and vascular patients — a key failure mode for **Model A**.

In [ ]:
miss_bl = df.groupby('surgery_type')['blood_loss_ml'].apply(
    lambda x: x.isnull().mean()).rename('missing_rate')
mean_observed = df.groupby('surgery_type')['blood_loss_ml'].mean().rename('Observed (true)')
mean_imputed  = df.groupby('surgery_type')['blood_loss_imputed'].mean().rename('Imputed (mean)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

miss_bl.sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='darkorange', edgecolor='white')
axes[0].set_title('Blood loss missingness\nby surgery type')
axes[0].set_ylabel('Missing rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].tick_params(axis='x', rotation=20)

pd.DataFrame({'Observed (true)': mean_observed, 'Imputed (mean)': mean_imputed}).plot(
    kind='bar', ax=axes[1], color=['tomato', 'steelblue'], edgecolor='white')
axes[1].set_title('Mean blood loss: true vs imputed')
axes[1].set_ylabel('Blood loss (mL)')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Blood loss MNAR — mean imputation underestimates cardiac/vascular risk', fontsize=10)
plt.tight_layout(); plt.show()

print('Blood loss missingness by surgery type:')
for surg, rate in miss_bl.sort_values(ascending=False).items():
    print(f'  {surg:14s}: {rate:.1%} missing')

**Expected output — Section 4:**

```
Blood loss missingness by surgery type:
  cardiac     : 50.9% missing   ← highest
  vascular    : 45.2% missing
  abdominal   : 23.9% missing
  orthopaedic : 23.6% missing
```

The right chart compares true (observed) vs imputed mean blood loss per surgery type. Cardiac and vascular bars will show a visible gap — imputed values are lower than true values, meaning the model gets an underestimate of how severe these cases are.

---
## Section 5 — Vital Signs Overview

Vital sign aggregates are available for all patients (no missing values).

Key pattern — **vascular haemodynamic masking**: vascular patients receive post-operative
beta-blockade, keeping heart rate near-normal (~70 bpm) even when 30-day complication risk
is elevated. A model relying heavily on vitals will systematically underestimate risk for
this subgroup. ICU lactate is also suppressed for vascular patients (~0.9 vs ~2.5 in others).

In [ ]:
vital_cols = ['hr_mean', 'rr_mean', 'spo2_mean', 'sbp_mean', 'temp_mean']

print('Vital sign means by outcome (major_complication_30d):')
print(
    df.groupby('major_complication_30d')[vital_cols]
    .mean().round(2)
    .rename(index={0: 'No complication', 1: 'Complication'})
    .T.to_string()
)
print()
print('Haemodynamic masking — HR mean and ICU lactate by surgery type:')
print(df.groupby('surgery_type')[['hr_mean', 'icu_lactate']].mean().round(2).to_string())

**Expected output — Section 5:**

```
Vital sign means by outcome (major_complication_30d):
           No complication  Complication
hr_mean              75.85         84.71
rr_mean              14.01         16.29
spo2_mean            98.17         97.12
sbp_mean            123.78        116.93
temp_mean            36.82         37.08

Haemodynamic masking — HR mean and ICU lactate by surgery type:
             hr_mean  icu_lactate
abdominal      77.22         2.23
cardiac        85.82         2.86
orthopaedic    78.19         2.35
vascular       70.13         0.95   ← lowest HR and lowest lactate
```

> **Key insight:** Vascular patients have the lowest heart rate (70 bpm) and the lowest ICU lactate (0.95) despite elevated complication risk. Beta-blockade suppresses both signals. A model that treats elevated HR as the main danger signal will systematically under-flag vascular patients.

---
## Section 6 — Data Profile Verdict

Answer these questions before running the app. Your answers inform the deployment
verdicts in Tab 5 (Report Builder).

---

### 6.1 — Note Missingness (Model C MNAR)

*Which surgery type has the worst note missingness? What does this mean for a model
that uses notes as a primary signal? Is note absence a signal of lower risk?*


---

### 6.2 — Blood Loss Imputation (Model A MNAR)

*Why does mean imputation of blood loss underestimate risk for cardiac/vascular patients?
What does `blood_loss_missing = 1` tell the model that `blood_loss_imputed` does not?*


---

### 6.3 — Vascular Haemodynamic Masking

*Why do vascular patients show near-normal HR and lactate despite elevated complication risk?
Which model architecture is most vulnerable to this, and why?*


---

**Next:** Run `01_preprocess_and_split.ipynb`.